# CSCI 5622 HW #4 - Questions (a) and (b)

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import BertTokenizer, BertModel
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
import os

## (a) Data processing

In [3]:
labels_path = r"../Study 1 (EDAIC)/DepressionLabels.csv"
transcripts_path = r"../Study 1 (EDAIC)/EDAIC Transcripts"

# Read depression labels
labels_df = pd.read_csv(labels_path)
labels_df

,Participant_ID,PHQ_Score
0,300,2
1,301,3
2,302,4
3,303,0
4,304,6
...,...,...
214,698,19
215,702,0
216,703,8
217,707,1


### Identify participants for which data is available

In [45]:
# Check that labels provided have a matching transcript
# Disregard participant data where a label/score or transcript are unavailable
ids = list()
transcripts_list = list()
phq_scores = list()
all_scores = labels_df['PHQ_Score'].to_numpy()
for i, id in enumerate(labels_df['Participant_ID'].to_numpy()):
    if os.path.isfile(f'{transcripts_path}/{str(id)}_Transcript.csv'):
        ids.append(id)
        transcripts_list.append(f'{str(id)}_Transcript.csv')
        phq_scores.append(all_scores[i])
print(len(ids))

134


### Gather language features for each transcript

In [5]:
def read_transcript_csv(path):
    with open(path, 'r') as f:
        lines = f.readlines()
    processed_lines = [l.strip() for l in lines[1:]]
    return processed_lines

def compute_transcript_sentiment(lines:list):
    '''
    Code written using Microsoft Copilot
    Calculates average of each sentiment score as given by the Vader sentiment package
    '''
    analyzer = SentimentIntensityAnalyzer()
    scores = [analyzer.polarity_scores(statement) for statement in lines]
    
    # Aggregate by mean
    avg_compound = sum(s['compound'] for s in scores) / len(scores)
    avg_pos = sum(s['pos'] for s in scores) / len(scores)
    avg_neg = sum(s['neg'] for s in scores) / len(scores)
    avg_neu = sum(s['neu'] for s in scores) / len(scores)
    
    return {
        'avg_compound': avg_compound,
        'avg_pos': avg_pos,
        'avg_neg': avg_neg,
        'avg_neu': avg_neu
    }

def get_transcript_embeddings(lines:list):
    '''
    Code written using Microsoft Copilot
    Get average embedding of a transcript using pretrained BERT by mean pooling all statement embeddings 
    '''
    # Define BERT model
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased')

    embeddings = list()
    for statement in lines: # Get embedding for each statement
        inputs = tokenizer(statement, return_tensors='pt', truncation=True, padding=True)
        with torch.no_grad():
            outputs = model(**inputs)
        # Get [CLS] token embedding (first token)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # shape: [1, hidden_size]
        embeddings.append(cls_embedding)  

    # Calculate average embedding across all statements
    transcript_embedding = torch.mean(torch.cat(embeddings, dim=0), dim=0)
    return transcript_embedding

#### Extract sentiment scores and BERT embeddings

In [13]:
# Iterate through all relevant CSV files
lang_features = dict()
all_transcripts = list()
for file in transcripts_list:
    # Read transcript and prepare for simple vectorization
    ls = read_transcript_csv(os.path.join(transcripts_path, file))
    all_transcripts.append(ls)
    
    # Get average sentiment scores using Vader
    sentiment_scores = compute_transcript_sentiment(ls)
    
    # Extract average embedding for each transcript using pretrained BERT
    embedding = get_transcript_embeddings(ls).numpy()

    # Store language features based on ID
    id = int(file[:3])
    tmp_features = {"sentiment":sentiment_scores, "embedding":embedding}
    lang_features[id] = tmp_features

#### Get syntactic vector of all transcripts

In [59]:
# Consolidate transcripts as contiguous lists (documents)
transcripts_concat = [' '.join(t) for t in all_transcripts]

# Vectorize
vectorizer = TfidfVectorizer(min_df=10)  # Reduce sparsity by requiring words appear in at least 10 documents
tfidf_matrix = vectorizer.fit_transform(transcripts_concat)
pure_tfidf_matrix = tfidf_matrix.toarray()

# Combine results with previous language features
for idx, id_ in enumerate(ids):
    # Add to the corresponding dictionary
    lang_features[id_]['tfidf'] = pure_tfidf_matrix[idx]

In [60]:
print(pure_tfidf_matrix.shape)

(134, 1005)


#### Combine data rows to save features as CSV

In [61]:
# Copilot assisted code generation
rows = list()
for i, id in enumerate(ids):
    features = lang_features[id]
    row = {'id':id}

    # Add PHQ scores (outcome)
    row['PHQ_Score'] = phq_scores[i]

    # Get sentiments
    for name,score in features['sentiment'].items():
        row[name] = score
    
    # Get embeddings
    for i, val in enumerate(features['embedding']):
        row[f'embed_{i}'] = val
    
    # Get tfidf features
    for i, val in enumerate(features['tfidf']):
        row[f'tfidf_{i}'] = val
    
    rows.append(row)

# Save language features as csv
lang_df = pd.DataFrame(rows)
lang_df.to_csv('edaic_transcript_features.csv', index=False)

## (b) Estimate depression severity using a decision tree

In [129]:
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.model_selection import KFold, train_test_split
from scipy.stats import pearsonr

### Define metrics

Pearson's correlation and absolute relative error (RE)

In [67]:
print(max(lang_df['PHQ_Score']))

22


In [148]:
def calc_re(pred_score, act_score, max_score=22):
    return abs(pred_score - act_score) / max_score

def calc_avg_re(y_preds, y_acts, max_score=22):
    avg_re = 0
    for pred, act in zip(y_preds, y_acts):
        avg_re += calc_re(pred, act)
    return avg_re / len(y_preds)

def calc_tot_re(y_preds, y_acts, max_score=22):
    tot_re = 0
    for pred, act in zip(y_preds, y_acts):
        tot_re += calc_re(pred, act)
    return tot_re

### Split data into 5 folds as instructed

In [128]:
X = lang_df.drop(columns=['id','PHQ_Score'])
y = lang_df['PHQ_Score']

# Split data into 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=59)
folds = list(kf.split(X))

# Reserve last fold as test set
train_folds = folds[:-1]
test_fold = folds[-1]

print(folds[3])

(array([  0,   1,   2,   3,   5,   6,   7,   8,  10,  11,  12,  13,  15,
        16,  18,  19,  20,  22,  23,  24,  25,  26,  27,  28,  30,  31,
        33,  34,  36,  38,  42,  43,  45,  46,  47,  48,  49,  50,  52,
        53,  56,  57,  58,  60,  61,  62,  63,  64,  65,  66,  67,  68,
        69,  70,  71,  72,  73,  75,  77,  79,  80,  81,  82,  83,  85,
        86,  88,  89,  90,  91,  92,  93,  94,  95,  96,  98, 100, 101,
       102, 103, 104, 105, 106, 107, 108, 109, 111, 112, 113, 114, 115,
       116, 117, 118, 119, 121, 122, 123, 124, 125, 126, 127, 128, 130,
       131, 132, 133]), array([  4,   9,  14,  17,  21,  29,  32,  35,  37,  39,  40,  41,  44,
        51,  54,  55,  59,  74,  76,  78,  84,  87,  97,  99, 110, 120,
       129]))


### Train and cross validate decision trees and evaluate predictions

In [ ]:
# Microsoft Copilot written decision tree cross validation
best_model = None
best_score = -1  # Pearson r ranges from -1 to 1, so start low

# Cross-validation using Pearson's r
for i, (train_idx, val_idx) in enumerate(train_folds):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = DecisionTreeRegressor(random_state=42)
    model.fit(X_tr, y_tr)
    
    preds = model.predict(X_val)
    
    # Compute Pearson correlation
    r, _ = pearsonr(y_val, preds)
    print(f"Fold {i+1} Pearson r: {r:.3f}")

    # Compute average relative error
    avg_re = 0
    for j, act_y in enumerate(y_val):
        avg_re += calc_re(preds[j], act_y)
    avg_re = avg_re / len(preds)
    print(f'Fold {i+1} Avg Relative Error: {avg_re:.3f}\n')

    # Track best model
    if r > best_score:
        best_score = r
        best_model = model

print(f"Best fold Pearson r: {best_score:.3f}")

# Test evaluation
test_idx = test_fold[1]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]
test_preds = best_model.predict(X_test)

test_r, _ = pearsonr(y_test, test_preds)
print(f"Test Pearson r: {test_r:.3f}")

Fold 1 Pearson r: 0.470
Fold 1 Avg Relative Error: 0.194

Fold 2 Pearson r: 0.392
Fold 2 Avg Relative Error: 0.175

Fold 3 Pearson r: -0.047
Fold 3 Avg Relative Error: 0.328

Fold 4 Pearson r: -0.119
Fold 4 Avg Relative Error: 0.273

Best fold Pearson r: 0.470
Test Pearson r: 1.000


In [149]:
# Split into train and test first
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=59)

# KFold only on training data
kf = KFold(n_splits=4, shuffle=True, random_state=59)

best_model = None
best_score = 1000

def calc_re(pred, actual):
    return abs(pred - actual) / actual if actual != 0 else 0

# Cross-validation on training set
for i, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

    model = DecisionTreeRegressor(random_state=42)
    model.fit(X_tr, y_tr)

    preds = model.predict(X_val)

    # Pearson correlation
    r, _ = pearsonr(y_val.values, preds)
    print(f"Fold {i+1} Pearson r: {r:.3f}")

    # Average relative error
    avg_re = calc_tot_re(preds, y_val.values)
    print(f"Fold {i+1} Avg Relative Error: {avg_re:.3f}\n")

    if avg_re < best_score:
        best_score = avg_re
        best_model = model

print(f"Best fold RE: {best_score:.3f}")

# Final evaluation on test set
test_preds = best_model.predict(X_test)
test_r, _ = pearsonr(y_test.values, test_preds)
test_re = calc_tot_re(test_preds, y_test.values)
print(f"Test Pearson r: {test_r:.3f}")
print(f"Test RE: {test_re:.3f}")
print(test_preds)
print(y_test.values)

Fold 1 Pearson r: -0.060
Fold 1 Avg Relative Error: 29.426

Fold 2 Pearson r: -0.207
Fold 2 Avg Relative Error: 46.994

Fold 3 Pearson r: 0.392
Fold 3 Avg Relative Error: 34.984

Fold 4 Pearson r: -0.041
Fold 4 Avg Relative Error: 33.367

Best fold RE: 29.426
Test Pearson r: 0.252
Test RE: 18.187
[ 7.  0.  0.  7.  1.  0.  1.  6.  2.  3.  6.  3.  4. 16.  1.  0.  3.  0.
  2.  2.  0.  0.  1.  2.  2.  3.  0.]
[ 0  0  0  2  3 10  0 10  9  3  3  1 11 17 12 13  7  3  8 22  3  4  0  1
  0  0  9]


In [ ]:
plt.figure(figsize=(12, 8))
plot_tree(best_model, feature_names=X.columns, class_names=True, filled=True)
plt.show()